# Условная диффузия: MLP[x, yₜ], 8 шагов

Та же одномерная функция и точки. Архитектура `Linear(2, 512) → ReLU → Linear(512, 1)`: кондишон `x` конкатенируется с зашумлённым значением `y_t`. Все параметры обучаются совместно. 2049 параметров вместо 1537 у обычной MLP той же ширины — добавилось 512 весов одного нового входа.

**Обучение:** для каждой точки на каждом обновлении выбираем случайный `t` из `1..8` и Gaussian-шум, затем строим

$$y_t=\sqrt{\bar\alpha_t}\,y_0+\sqrt{1-\bar\alpha_t}\,\varepsilon,\quad \varepsilon\sim\mathcal N(0,1).$$

Сеть предсказывает **чистый y₀**, лосс — MSE с исходным таргетом. На одно обновление приходится один forward/backward; уровни шума перемешиваются на протяжении всего обучения. Это не расписание убывания шума по эпохам.

**Семплинг:** начинаем с `y_8 ~ N(0,1)` и делаем ровно **8 DDIM-шагов** (`eta=0`) одной и той же сетью, удерживая `x` неизменным. Чистый таргет семплеру не передаётся. Косинусное расписание имеет `alpha_bar[0]=1`, `alpha_bar[8]=0`, поэтому начальное состояние — чистый шум, а последний шаг — чистое предсказание.

Чтобы сохранить ровно два входа, **t не подаётся в MLP** и используется только шедулером. Это упрощённый вариант с x₀-prediction для детерминированной зависимости `y₀=f(x)`, а не epsilon-prediction с time embedding.

Вверху — MSE результата полного 8-шагового семплинга, внизу — этот результат и целевая функция. Начальный шум для evaluation фиксирован между кадрами. Обучение работает в фоне; все 8 шагов evaluation выполняет рендерер на свежей копии модели по запросу браузера, без очереди кадров.

Формула семплера: [DDIM](https://arxiv.org/abs/2010.02502); косинусное расписание: [Improved DDPM](https://arxiv.org/abs/2102.09672).

In [ ]:
from pathlib import Path
import sys

project_root = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "notebooks" / "_shared").is_dir()
)
shared_parent = str(project_root / "notebooks")
if shared_parent not in sys.path:
    sys.path.insert(0, shared_parent)

import torch
from _shared import Config
from _shared.conditional_diffusion import DiffusionConfig
from _shared.diffusion_experiment import DiffusionExperiment
from _shared.visualization import train_live

torch.set_num_threads(1)

In [12]:
config = Config(
    seed=42,
    group_sizes=(256,),
    steps_per_stage=10_000,
    learning_rate=0.01,
    n_train=256,
    n_grid=601,
    x_min=-3.0,
    x_max=3.0,
)
diffusion_config = DiffusionConfig(
    denoise_steps=8,
    cosine_offset=0.008,
)
target_fps = 10

In [13]:
diffusion_experiment = DiffusionExperiment(config, diffusion_config)
await train_live(diffusion_experiment, target_fps=target_fps)

In [14]:
# Экстраполяция без переобучения: padding с каждой стороны интервала.
from copy import deepcopy
import plotly.graph_objects as go
from IPython.display import display
from _shared.experiment import target_function

padding_fraction = 0.10
fit_config = diffusion_experiment.config
padding = (fit_config.x_max - fit_config.x_min) * padding_fraction
n_pad = max(0, round((fit_config.n_grid - 1) * padding_fraction))

# Берём согласованную копию, не меняя обученную модель.
with diffusion_experiment.model_lock:
    extrapolation_model = deepcopy(diffusion_experiment.model)
extrapolation_model.eval()
reference = extrapolation_model.hidden.weight
grid_kwargs = dict(device=reference.device, dtype=reference.dtype)
left_x = torch.linspace(
    fit_config.x_min - padding, fit_config.x_min, n_pad + 1, **grid_kwargs
)[:-1, None]
right_x = torch.linspace(
    fit_config.x_max, fit_config.x_max + padding, n_pad + 1, **grid_kwargs
)[1:, None]
core_x = diffusion_experiment.x_grid.to(**grid_kwargs)
extrapolation_x = torch.cat((left_x, core_x, right_x), dim=0)

# Внутри интервала сохраняем тот же стартовый шум, что на живом графике.
noise_generator = torch.Generator(device=reference.device).manual_seed(
    fit_config.seed + 30000
)
left_noise = torch.randn(left_x.shape, generator=noise_generator, **grid_kwargs)
right_noise = torch.randn(right_x.shape, generator=noise_generator, **grid_kwargs)
core_noise = diffusion_experiment.eval_noise[fit_config.n_train :].to(**grid_kwargs)
extrapolation_noise = torch.cat((left_noise, core_noise, right_noise), dim=0)
with torch.inference_mode():
    extrapolation_prediction = diffusion_experiment.diffusion.sample(
        extrapolation_model, extrapolation_x, initial_noise=extrapolation_noise
    )
    extrapolation_target = target_function(extrapolation_x)

plot_x = extrapolation_x.squeeze(1).cpu().tolist()
extrapolation_figure = go.Figure(
    [
        go.Scatter(
            x=plot_x,
            y=extrapolation_target.squeeze(1).cpu().tolist(),
            name="Целевая функция",
            mode="lines",
            line=dict(color="#64748b", width=3),
        ),
        go.Scatter(
            x=plot_x,
            y=extrapolation_prediction.squeeze(1).cpu().tolist(),
            name="Diffusion MLP",
            mode="lines",
            line=dict(color="#2563eb", width=3),
        ),
    ]
)
for x0, x1 in (
    (fit_config.x_min - padding, fit_config.x_min),
    (fit_config.x_max, fit_config.x_max + padding),
):
    extrapolation_figure.add_vrect(
        x0=x0, x1=x1, fillcolor="#f59e0b", opacity=0.12, line_width=0, layer="below"
    )
for boundary in (fit_config.x_min, fit_config.x_max):
    extrapolation_figure.add_vline(x=boundary, line_dash="dash", line_color="#f59e0b")
extrapolation_figure.update_layout(
    title=f"Экстраполяция · padding {padding_fraction:.0%} с каждой стороны · {diffusion_experiment.diffusion.config.denoise_steps} DDIM-шагов",
    template="plotly_white",
    autosize=True,
    height=550,
    margin=dict(l=65, r=25, t=60, b=65),
    legend=dict(orientation="h", y=-0.15),
    xaxis=dict(
        title="x", range=[fit_config.x_min - padding, fit_config.x_max + padding]
    ),
    yaxis=dict(title="f(x)"),
)
payload = extrapolation_figure.to_plotly_json()
payload["config"] = dict(responsive=True, displayModeBar=False)
display({"application/vnd.plotly.v1+json": payload}, raw=True)